# LLM Fine-Tuning Pipeline

## Objective

Reproduce the resume project: supervised fine-tune a LLaMA 3 8B model on a 50K-sample domain dataset with LLaMA-Factory, LoRA/QLoRA, mixed precision, data preprocessing, automated evaluation, and FastAPI/Docker serving artifacts.

Recommended runtime: Colab Pro A100/L4 or Kaggle GPU. Use a Hugging Face token with access to the selected base model.

## Colab dependency compatibility

Colab currently starts with Gradio 6.x, while LLaMA-Factory installs its supported Gradio 5.50.0 stack. Gradio 6.20.0 requires hf-gradio, so the setup cell keeps that package until LLaMA-Factory has replaced Gradio 6.x. It then removes hf-gradio, whose Gradio Client 2.x requirement is incompatible with LLaMA-Factory's Gradio Client 1.14.0.

The final project environment pins Pydantic 2.12.3, Gradio 5.50.0, and Gradio Client 1.14.0. Unrelated Colab applications with conflicting Pydantic or Starlette requirements are removed.

Do not upgrade Pydantic or Starlette after this cell. If Colab asks for a runtime restart after installation, use **Runtime > Restart session** once, then rerun the setup and environment-validation cells.

In [14]:
# Use one exact constraint for every install command.
from pathlib import Path
CONSTRAINTS = Path("/content/project1-constraints.txt")
CONSTRAINTS.write_text(
    "pydantic==2.12.3\n"
    "gradio==5.50.0\n"
    "gradio-client==1.14.0\n"
)

# Keep hf-gradio until Gradio 6.x has been replaced to avoid a transient conflict.
!python -m pip uninstall -y -q google-adk google-genai python-fasthtml fastai fastprogress
!python -m pip install -q -c /content/project1-constraints.txt "pydantic==2.12.3"

!test -d /content/LlamaFactory/.git || git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
%cd /content/LlamaFactory
!python -m pip install -q uv
!uv pip install --system --constraint /content/project1-constraints.txt -e .
!uv pip install --system --constraint /content/project1-constraints.txt \
    bitsandbytes accelerate peft trl datasets evaluate rouge-score \
    bert-score fastapi uvicorn jedi

# Gradio 5.50.0 does not use hf-gradio, whose client requirement is for Gradio 6.x.
!python -m pip uninstall -y -q hf-gradio
!python -m pip check
!llamafactory-cli version

/content/LlamaFactory
Using Python 3.12.13 environment at: /usr
Resolved 127 packages in 377ms                                       
Prepared 1 package in 161ms                                              
Uninstalled 1 package in 0.45ms
Installed 1 package in 1ms.dev0 (from file:///content/LlamaF
 ~ llamafactory==0.9.6.dev0 (from file:///content/LlamaFactory)
Using Python 3.12.13 environment at: /usr
Checked 11 packages in 99ms
fastdownload 0.0.7 requires fastprogress, which is not installed.
----------------------------------------------------------
| Welcome to LLaMA Factory, version 0.9.6.dev0           |
|                                                        |
| Project page: https://github.com/hiyouga/LLaMA-Factory |
----------------------------------------------------------


In [15]:
from importlib.metadata import version

for package in ["llamafactory", "fastapi", "starlette", "pydantic",
                "gradio", "gradio-client", "transformers", "peft", "bitsandbytes"]:
    print(f"{package:14s} {version(package)}")

import fastapi, starlette, pydantic
assert pydantic.__version__ == "2.12.3"
assert version("gradio") == "5.50.0"
assert version("gradio-client") == "1.14.0"
print("Core imports and pinned dependency stack: OK")

llamafactory   0.9.6.dev0
fastapi        0.139.0
starlette      0.52.1
pydantic       2.12.3
gradio         5.50.0
gradio-client  1.14.0
transformers   5.8.0
peft           0.18.1
bitsandbytes   0.49.2
Core imports and pinned dependency stack: OK


In [ ]:
# Authenticate for the gated Meta Llama model without storing a token here.
# vs code can't access colab secrets so have to use the huggingface_hub login() prompt with manual verification.
import os, json, random, time, statistics, pathlib
from dataclasses import dataclass
from huggingface_hub import HfApi, get_token, login

PROJECT_DIR = pathlib.Path('/content/fine_tuning_pipeline')
DATA_DIR = PROJECT_DIR / 'data'
OUT_DIR = PROJECT_DIR / 'outputs'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'  # change if access differs
DATASET_JSONL = DATA_DIR / 'domain_sft_50k.jsonl'
EVAL_JSONL = DATA_DIR / 'eval_500.jsonl'
SEED = 42
random.seed(SEED)

# Colab Secrets cannot be fetched through the VS Code Colab extension.
# login() prompts securely and caches the credential in this runtime.
hf_token = os.getenv("HF_TOKEN") or get_token()
if not hf_token:
    print("Opening Hugging Face authentication...")
    login(add_to_git_credential=False, skip_if_logged_in=False)
    hf_token = get_token()

if not hf_token:
    raise RuntimeError(
        "Hugging Face login did not return a token. Run `hf auth login` in "
        "the VS Code terminal attached to this Colab runtime, then rerun this cell."
    )

os.environ["HF_TOKEN"] = hf_token

try:
    model_info = HfApi(token=hf_token).model_info(MODEL_NAME)
except Exception as exc:
    raise RuntimeError(
        "Hugging Face authentication succeeded, but this account cannot access "
        f"{MODEL_NAME}. Accept the model terms with the same account and verify "
        "that its token has read access."
    ) from exc

MODEL_REVISION = model_info.sha
print("Hugging Face access verified:", MODEL_NAME, MODEL_REVISION[:12])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Opening Hugging Face authentication...


Hugging Face access verified: meta-llama/Meta-Llama-3-8B-Instruct 8afb486c1db2


## 1. Prepare or load the 50K JSONL dataset

Required schema per line:

```json
{"instruction": "...", "input": "...", "output": "...", "metadata": {"topic": "..."}}
```

Replace the synthetic bootstrap below with your approved domain dataset. Keep the synthetic generator only for smoke tests.

In [22]:
'''
def make_synthetic_example(i):
    topic = ['RAG', 'agent tooling', 'model serving', 'evaluation'][i % 4]
    return {
        'instruction': f'Answer as an AI platform engineer about {topic}.',
        'input': f'Question {i}: What is a best practice for {topic}?',
        'output': f'A strong practice for {topic} is to define a baseline, control variables, measure outcomes, and document tradeoffs.',
        'metadata': {'topic': topic, 'synthetic': True}
    }

if not DATASET_JSONL.exists():
    with DATASET_JSONL.open('w') as f:
        for i in range(50000):
            f.write(json.dumps(make_synthetic_example(i)) + '\n')
print(DATASET_JSONL, DATASET_JSONL.exists())
'''

%pip install -q beautifulsoup4

import json
from bs4 import BeautifulSoup
from datasets import load_dataset


TARGET = 50_500  # 50K training + 500 evaluation
REQUIRED_FIELDS = {"instruction", "input", "output", "metadata"}

def html_to_text(value):
    return BeautifulSoup(value, "html.parser").get_text("\n", strip=True)

def validate_cached_jsonl(path, expected_rows):
    if not path.exists():
        return False, 0

    try:
        with path.open(encoding="utf-8") as source:
            row_count = 0
            for line in source:
                if not line.strip():
                    continue
                record = json.loads(line)
                if not REQUIRED_FIELDS.issubset(record):
                    return False, row_count
                row_count += 1
        return row_count == expected_rows, row_count
    except (OSError, json.JSONDecodeError):
        return False, 0

cache_valid, cached_rows = validate_cached_jsonl(DATASET_JSONL, TARGET)

if cache_valid:
    print(f"Using cached dataset: {DATASET_JSONL} ({cached_rows:,} rows)")
else:
    if DATASET_JSONL.exists():
        print(f"Replacing incomplete cache: {cached_rows:,}/{TARGET:,} rows")
    else:
        print(f"No cache found; downloading {TARGET:,} rows once...")

    DATASET_JSONL.parent.mkdir(parents=True, exist_ok=True)
    temporary_jsonl = DATASET_JSONL.with_suffix(".jsonl.incomplete")
    temporary_jsonl.unlink(missing_ok=True)

    stream = load_dataset(
        "HuggingFaceH4/stack-exchange-preferences",
        data_dir="data/Stackoverflow.com",
        split="train",
        streaming=True,
    ).shuffle(seed=SEED, buffer_size=20_000)

    saved_rows = 0
    with temporary_jsonl.open("w", encoding="utf-8") as output:
        for row in stream:
            candidates = [
                answer for answer in row["answers"]
                if answer["pm_score"] >= 2
            ]
            if not candidates:
                continue

            best = max(
                candidates,
                key=lambda answer: (answer["selected"], answer["pm_score"]),
            )

            question = html_to_text(row["question"])
            answer = html_to_text(best["text"])

            if not (40 <= len(question) <= 8_000):
                continue
            if not (80 <= len(answer) <= 8_000):
                continue

            record = {
                "instruction": (
                    "Answer the following software-engineering question accurately. "
                    "Explain the reasoning and include code when appropriate."
                ),
                "input": question,
                "output": answer,
                "metadata": {
                    "source_url": row["metadata"][0],
                    "answer_author": best.get("author"),
                    "author_profile": best.get("author_profile"),
                    "score": best["pm_score"],
                    "accepted": best["selected"],
                    "license": "CC BY-SA 4.0",
                },
            }
            output.write(json.dumps(record, ensure_ascii=False) + "\n")
            saved_rows += 1

            if saved_rows == TARGET:
                break

    if saved_rows != TARGET:
        temporary_jsonl.unlink(missing_ok=True)
        raise RuntimeError(f"Prepared only {saved_rows:,}/{TARGET:,} valid rows")

    temporary_jsonl.replace(DATASET_JSONL)
    print(f"Saved dataset cache: {DATASET_JSONL} ({saved_rows:,} rows)")

Replacing incomplete cache: 0/50,500 rows


Resolving data files:   0%|          | 0/335 [00:00<?, ?it/s]

Saved dataset cache: /content/fine_tuning_pipeline/data/domain_sft_50k.jsonl (50,500 rows)


In [23]:
def load_jsonl(path, limit=None):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
                if limit and len(rows) >= limit:
                    break
    return rows

rows = load_jsonl(DATASET_JSONL)
print('raw rows:', len(rows))
print(rows[0])

raw rows: 50500
{'instruction': 'Answer the following software-engineering question accurately. Explain the reasoning and include code when appropriate.', 'input': "I am trying to insert/update post meta when the user registers. Before writing that action, I am testing this code on the page so whenever the page refresh it will insert/update the post meta.\nQuestion:\nHowever, the below code is not inserting/updating anything in post\n  meta. Can anyone tell me what is wrong in this code or how to fix it?\n$groupItem = get_post(123);\n\nif ($groupItem && $groupItem->post_type == 'cpt_group') {\n\n    $meta     = 'group_users';\n    $user_ids = get_post_meta($groupItem->ID, $meta, TRUE);\n\n    if ( ! $user_ids) {\n        $user_ids = [];\n        add_post_meta($groupItem->ID, $meta, array_push($user_ids, 26));\n    } else {\n        update_post_meta($groupItem->ID, $meta, array_push($user_ids, 26), $user_ids);\n    }\n}", 'output': "The purpose of DTO is used to pack some data to transf

## 2. Data quality gates

These gates make the quality and GPU-hour claims credible: remove bad samples before paying for training.

In [24]:
def clean_rows(rows, max_chars=8000):
    seen = set()
    clean = []
    rejected = {'duplicate': 0, 'empty': 0, 'too_long': 0}
    for r in rows:
        inst = (r.get('instruction') or '').strip()
        inp = (r.get('input') or '').strip()
        out = (r.get('output') or '').strip()
        key = (inst.lower(), inp.lower(), out.lower())
        if not inst or not out:
            rejected['empty'] += 1
            continue
        if len(inst) + len(inp) + len(out) > max_chars:
            rejected['too_long'] += 1
            continue
        if key in seen:
            rejected['duplicate'] += 1
            continue
        seen.add(key)
        clean.append({'instruction': inst, 'input': inp, 'output': out, 'metadata': r.get('metadata', {})})
    return clean, rejected

clean, rejected = clean_rows(rows)
random.shuffle(clean)
eval_rows = clean[:500]
train_rows = clean[500:]
print({'train': len(train_rows), 'eval': len(eval_rows), 'rejected': rejected})

{'train': 49693, 'eval': 500, 'rejected': {'duplicate': 0, 'empty': 0, 'too_long': 307}}


In [25]:
TRAIN_JSON = DATA_DIR / 'train_llamafactory.json'
EVAL_JSON = DATA_DIR / 'eval_llamafactory.json'

def to_llamafactory(row):
    return {
        'instruction': row['instruction'],
        'input': row.get('input', ''),
        'output': row['output'],
    }

TRAIN_JSON.write_text(json.dumps([to_llamafactory(r) for r in train_rows], indent=2))
EVAL_JSON.write_text(json.dumps([to_llamafactory(r) for r in eval_rows], indent=2))
print(TRAIN_JSON, EVAL_JSON)

/content/fine_tuning_pipeline/data/train_llamafactory.json /content/fine_tuning_pipeline/data/eval_llamafactory.json


## 3. Register dataset in LLaMA-Factory

This appends local datasets to `data/dataset_info.json`. If rerun, it updates the same keys.

In [26]:
info_path = pathlib.Path('data/dataset_info.json')
dataset_info = json.loads(info_path.read_text())
dataset_info['hongda_domain_sft'] = {
    'file_name': str(TRAIN_JSON),
    'columns': {'prompt': 'instruction', 'query': 'input', 'response': 'output'}
}
dataset_info['hongda_domain_eval'] = {
    'file_name': str(EVAL_JSON),
    'columns': {'prompt': 'instruction', 'query': 'input', 'response': 'output'}
}
info_path.write_text(json.dumps(dataset_info, indent=2))
print('registered datasets')

registered datasets


## 4. Train LoRA/QLoRA adapter

Use QLoRA on T4/L4. On A100, bf16 LoRA without 4-bit can also be tested. Start small for a smoke test, then scale to full run.

In [31]:
# For resume training only, check for existing processes and checkpoints.
!pgrep -af "llamafactory-cli|llamafactory.launcher"
!nvidia-smi
!find /content/fine_tuning_pipeline/outputs/llama3_lora_adapter \
  -maxdepth 1 -type d -name "checkpoint-*"

Wed Jul 22 04:56:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P0             45W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [32]:
SFT_YAML = OUT_DIR / 'llama3_lora_sft.yaml'
import torch
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
SFT_YAML.write_text(f'''
stage: sft
do_train: true
model_name_or_path: {MODEL_NAME}
dataset: hongda_domain_sft
template: llama3
finetuning_type: lora
lora_target: all
output_dir: {OUT_DIR / 'llama3_lora_adapter'}
overwrite_cache: true
overwrite_output_dir: true
cutoff_len: 2048
preprocessing_num_workers: 4
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
lr_scheduler_type: cosine
logging_steps: 10
save_steps: 50
save_total_limit: 2
learning_rate: 2.0e-4
num_train_epochs: 1.0
max_samples: 50000
plot_loss: true
bf16: {str(USE_BF16).lower()}
fp16: {str(not USE_BF16).lower()}
quantization_bit: 4
upcast_layernorm: true
gradient_checkpointing: true
packing: true
seed: {SEED}
'''.strip())
print(SFT_YAML.read_text())

stage: sft
do_train: true
model_name_or_path: meta-llama/Meta-Llama-3-8B-Instruct
dataset: hongda_domain_sft
template: llama3
finetuning_type: lora
lora_target: all
output_dir: /content/fine_tuning_pipeline/outputs/llama3_lora_adapter
overwrite_cache: true
overwrite_output_dir: true
cutoff_len: 2048
preprocessing_num_workers: 4
per_device_train_batch_size: 2
gradient_accumulation_steps: 8
lr_scheduler_type: cosine
logging_steps: 10
save_steps: 50
save_total_limit: 2
learning_rate: 2.0e-4
num_train_epochs: 1.0
max_samples: 50000
plot_loss: true
bf16: true
fp16: false
quantization_bit: 4
upcast_layernorm: true
gradient_checkpointing: true
packing: true
seed: 42


In [33]:
# Full training command. Uncomment after confirming GPU and HF access.
!llamafactory-cli train {SFT_YAML}
# print('Training command prepared. Uncomment the previous line for the full run.')
print("training ended")

[INFO|2026-07-22 04:57:12] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:780] 2026-07-22 04:57:12,958 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Meta-Llama-3-8B-Instruct/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2/config.json
[INFO|configuration_utils.py:856] 2026-07-22 04:57:12,963 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": 128009,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 8192,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,

## 5. Base vs tuned evaluation

**Primary criterion: answer-only perplexity on the fixed 500-example holdout.** It measures how well each model predicts the reference answer when given the same instruction and question. Lower perplexity is better. Only assistant-answer tokens contribute to the loss; prompt tokens are masked.

The comparison loads one 4-bit base model plus the trained LoRA adapter. The base pass temporarily disables that adapter, so model revision, quantization, tokenizer, prompts, truncation, and hardware are controlled. The report also includes paired-example win rate and a bootstrap 95% confidence interval for mean NLL improvement. Start with 100 examples for a practical A100 run; set MAX_EVAL_SAMPLES = 500 for the final portfolio result.

In [37]:
import gc
import math
import numpy as np
import torch
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# A failed prior evaluation can leave the 8B model resident on the GPU.
for stale_name in ("model", "base_model"):
    if stale_name in globals():
        del globals()[stale_name]
gc.collect()
torch.cuda.empty_cache()

ADAPTER_DIR = OUT_DIR / "llama3_lora_adapter"
EVAL_RESULTS_JSON = OUT_DIR / "base_vs_tuned_eval.json"
MAX_EVAL_SAMPLES = 100  # Use 500 for the final portfolio result.
EVAL_MAX_LENGTH = 2048  # Match training cutoff_len.
BOOTSTRAP_SAMPLES = 2_000

if not (ADAPTER_DIR / "adapter_config.json").exists():
    raise FileNotFoundError(f"Trained adapter not found at {ADAPTER_DIR}")
if not EVAL_JSON.exists():
    raise FileNotFoundError(f"Held-out evaluation file not found at {EVAL_JSON}")

eval_examples = json.loads(EVAL_JSON.read_text())
eval_rng = random.Random(SEED)
if len(eval_examples) > MAX_EVAL_SAMPLES:
    eval_examples = eval_rng.sample(eval_examples, MAX_EVAL_SAMPLES)

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer_source = ADAPTER_DIR if (ADAPTER_DIR / "tokenizer_config.json").exists() else MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_source,
    token=hf_token,
    revision=MODEL_REVISION if str(tokenizer_source) == MODEL_NAME else None,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    token=hf_token,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=compute_dtype,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
model.config.use_cache = False
model_device = next(model.parameters()).device

def infer_eval_seqlen(source_len, target_len, cutoff_len):
    # Match LLaMA-Factory's prompt/response-aware cutoff allocation.
    if target_len * 2 < cutoff_len:
        max_target_len = target_len
    elif source_len * 2 < cutoff_len:
        max_target_len = cutoff_len - source_len
    else:
        max_target_len = int(cutoff_len * target_len / (source_len + target_len))
    max_target_len = max(1, min(max_target_len, target_len))
    max_source_len = max(0, min(cutoff_len - max_target_len, source_len))
    return max_source_len, max_target_len

def to_token_id_list(encoded):
    # Transformers 5 fast tokenizers may return tokenizers.Encoding here.
    if hasattr(encoded, "ids"):
        encoded = encoded.ids
    elif hasattr(encoded, "input_ids"):
        encoded = encoded.input_ids
    elif isinstance(encoded, dict) and "input_ids" in encoded:
        encoded = encoded["input_ids"]

    if isinstance(encoded, torch.Tensor):
        encoded = encoded.detach().cpu().tolist()
    if encoded and isinstance(encoded[0], (list, tuple)):
        encoded = encoded[0]
    return [int(token_id) for token_id in encoded]

def encode_eval_example(example):
    user_content = example["instruction"].strip()
    if example.get("input", "").strip():
        user_content += "\n\n" + example["input"].strip()

    prompt_messages = [{"role": "user", "content": user_content}]
    full_messages = prompt_messages + [
        {"role": "assistant", "content": example["output"].strip()}
    ]
    prompt_ids = to_token_id_list(tokenizer.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True
    ))
    full_ids = to_token_id_list(tokenizer.apply_chat_template(
        full_messages, tokenize=True, add_generation_prompt=False
    ))

    if full_ids[:len(prompt_ids)] == prompt_ids:
        answer_ids = full_ids[len(prompt_ids):]
    else:
        # Conservative fallback for a tokenizer whose chat template changes the prefix.
        answer_ids = to_token_id_list(
            tokenizer.encode(example["output"].strip(), add_special_tokens=False)
        )
        if tokenizer.eos_token_id is not None:
            answer_ids.append(tokenizer.eos_token_id)

    source_len, target_len = infer_eval_seqlen(
        len(prompt_ids), len(answer_ids), EVAL_MAX_LENGTH
    )
    prompt_ids = prompt_ids[:source_len]
    answer_ids = answer_ids[:target_len]
    input_ids = prompt_ids + answer_ids
    labels = [-100] * source_len + answer_ids
    answer_tokens = sum(label != -100 for label in labels[1:])
    if answer_tokens == 0:
        return None

    return {
        "input_ids": torch.tensor([input_ids], device=model_device),
        "attention_mask": torch.ones((1, len(input_ids)), dtype=torch.long, device=model_device),
        "labels": torch.tensor([labels], device=model_device),
        "answer_tokens": answer_tokens,
    }

def forward_nll(batch, adapter_enabled):
    model_inputs = {
        "input_ids": batch["input_ids"],
        "attention_mask": batch["attention_mask"],
        "labels": batch["labels"],
        "use_cache": False,
    }
    with torch.inference_mode():
        if adapter_enabled:
            output = model(**model_inputs)
        else:
            with model.disable_adapter():
                output = model(**model_inputs)
    return float(output.loss.detach().cpu())

per_example = []
for index, example in enumerate(tqdm(eval_examples, desc="Base vs tuned evaluation")):
    batch = encode_eval_example(example)
    if batch is None:
        continue

    base_nll = forward_nll(batch, adapter_enabled=False)
    tuned_nll = forward_nll(batch, adapter_enabled=True)
    per_example.append({
        "sample_index": index,
        "answer_tokens": batch["answer_tokens"],
        "base_nll": base_nll,
        "tuned_nll": tuned_nll,
    })

if not per_example:
    raise RuntimeError("No evaluation examples contained answer tokens within the cutoff length")

total_answer_tokens = sum(row["answer_tokens"] for row in per_example)
base_nll = sum(row["base_nll"] * row["answer_tokens"] for row in per_example) / total_answer_tokens
tuned_nll = sum(row["tuned_nll"] * row["answer_tokens"] for row in per_example) / total_answer_tokens
base_perplexity = math.exp(base_nll)
tuned_perplexity = math.exp(tuned_nll)
perplexity_reduction_pct = 100 * (base_perplexity - tuned_perplexity) / base_perplexity

paired_improvements = np.array([
    row["base_nll"] - row["tuned_nll"] for row in per_example
])
win_rate_pct = 100 * float(np.mean(paired_improvements > 0))
bootstrap_rng = np.random.default_rng(SEED)
bootstrap_means = np.empty(BOOTSTRAP_SAMPLES)
for bootstrap_index in range(BOOTSTRAP_SAMPLES):
    sample = bootstrap_rng.choice(paired_improvements, size=len(paired_improvements), replace=True)
    bootstrap_means[bootstrap_index] = sample.mean()
ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])

summary = {
    "criterion": "answer-only held-out perplexity (lower is better)",
    "model_revision": MODEL_REVISION,
    "adapter_dir": str(ADAPTER_DIR),
    "examples_scored": len(per_example),
    "answer_tokens_scored": total_answer_tokens,
    "base_answer_nll": round(base_nll, 6),
    "tuned_answer_nll": round(tuned_nll, 6),
    "base_perplexity": round(base_perplexity, 4),
    "tuned_perplexity": round(tuned_perplexity, 4),
    "perplexity_reduction_pct": round(perplexity_reduction_pct, 2),
    "tuned_example_win_rate_pct": round(win_rate_pct, 2),
    "mean_nll_improvement_95pct_ci": [round(float(ci_low), 6), round(float(ci_high), 6)],
}
EVAL_RESULTS_JSON.write_text(json.dumps({"summary": summary, "per_example": per_example}, indent=2))
print(json.dumps(summary, indent=2))
print("Saved:", EVAL_RESULTS_JSON)

del model, base_model
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Base vs tuned evaluation:   0%|          | 0/100 [00:00<?, ?it/s]

{
  "criterion": "answer-only held-out perplexity (lower is better)",
  "model_revision": "8afb486c1db24fe5011ec46dfbe5b5dccdb575c2",
  "adapter_dir": "/content/fine_tuning_pipeline/outputs/llama3_lora_adapter",
  "examples_scored": 100,
  "answer_tokens_scored": 18798,
  "base_answer_nll": 2.073928,
  "tuned_answer_nll": 1.711249,
  "base_perplexity": 7.956,
  "tuned_perplexity": 5.5359,
  "perplexity_reduction_pct": 30.42,
  "tuned_example_win_rate_pct": 100.0,
  "mean_nll_improvement_95pct_ci": [
    0.475145,
    0.590001
  ]
}
Saved: /content/fine_tuning_pipeline/outputs/base_vs_tuned_eval.json


## 6. FastAPI serving artifact

This writes a deployment skeleton. In production, build the image on a CUDA host and benchmark under identical prompt lengths.

In [ ]:
SERVE_DIR = PROJECT_DIR / 'serve'
SERVE_DIR.mkdir(exist_ok=True)
(SERVE_DIR / 'app.py').write_text('''
import time
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="LLaMA 3 LoRA SFT Service")

class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=8000)
    max_new_tokens: int = Field(default=256, ge=1, le=1024)

@app.on_event("startup")
def load_model():
    # Load tokenizer, base model, and LoRA adapter once here.
    app.state.ready = True

@app.get("/health")
def health():
    return {"ready": getattr(app.state, "ready", False)}

@app.post("/generate")
def generate(req: GenerateRequest):
    start = time.perf_counter()
    # Replace with model.generate(...)
    text = "Replace this placeholder with generated text."
    return {"text": text, "latency_ms": round((time.perf_counter() - start) * 1000, 2)}
''')
(SERVE_DIR / 'Dockerfile').write_text('''
FROM nvidia/cuda:12.1.1-runtime-ubuntu22.04
WORKDIR /app
RUN apt-get update && apt-get install -y python3-pip && rm -rf /var/lib/apt/lists/*
COPY requirements.txt .
RUN pip3 install --no-cache-dir -r requirements.txt
COPY app.py .
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
''')
(SERVE_DIR / 'requirements.txt').write_text('fastapi\nuvicorn[standard]\ntransformers\npeft\nbitsandbytes\naccelerate\ntorch\n')
print('wrote serving files to', SERVE_DIR)